# Applio One Click Gradio

กดรันเซลล์เดียวด้านล่าง แล้วรอจนขึ้นลิงก์ Gradio

In [ ]:
# @title กดรันครั้งเดียวเพื่อเปิด Applio Gradio
# @markdown ถ้า Colab ขอสิทธิ์ Google Drive ให้กดยืนยัน 1 ครั้ง แล้วปล่อยให้รันต่อ
mount_drive = True # @param {type:"boolean"}
sync_models_with_drive = True # @param {type:"boolean"}
sharing_method = "gradio" # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "" # @param {type:"string"}
tensorboard = False # @param {type:"boolean"}
language = "th_TH" # @param ["th_TH", "en_US"]

from IPython.display import clear_output
from pathlib import Path
import os

repo_url = "https://github.com/IAHispano/Applio/"
repo_name = "Applio"
repo_path = Path('/content') / repo_name
logs_path = repo_path / 'logs'
backups_path = Path('/content/drive/MyDrive/ApplioBackup')

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')

%cd /content
!git config --global advice.detachedHead false
if repo_path.exists():
    %cd /content/Applio
    !git pull
else:
    !git clone {repo_url}
    %cd /content/Applio

clear_output()
print('กำลังติดตั้งของจำเป็น รอสักครู่...')
!apt update -y
!apt install -y portaudio19-dev
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel > /dev/null 2>&1
!python core.py prerequisites --models --pretraineds-hifigan

if mount_drive and sync_models_with_drive and Path('/content/drive').is_mount():
    print('กำลังเชื่อมโฟลเดอร์โมเดลกับ Google Drive...')
    backups_path.mkdir(parents=True, exist_ok=True)
    logs_path.mkdir(parents=True, exist_ok=True)
    temp_names = {'mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2'}
    for item in list(logs_path.iterdir()):
        if item.name not in temp_names and item.is_dir() and not item.is_symlink():
            target = backups_path / item.name
            if not target.exists():
                !mv "{item}" "{target}"
    for item in list(backups_path.iterdir()):
        link = logs_path / item.name
        if item.name not in temp_names and not link.exists():
            !ln -s "{item}" "{link}"

for name in ['mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2']:
    path = logs_path / name
    if path.exists() and (path.is_symlink() or not path.is_dir()):
        !rm -rf "{path}"
    path.mkdir(parents=True, exist_ok=True)

clear_output()
print('ติดตั้งเสร็จแล้ว กำลังเปิด Gradio...')

# ตั้งภาษาหน้าเว็บ (Applio มีภาษาไทยในตัว: assets/i18n/languages/th_TH.json)
import json
os.environ['LANG'] = f'{language}.UTF-8'
config_path = repo_path / 'assets' / 'config.json'
template_path = repo_path / 'assets' / 'config_template.json'
if not config_path.exists() and template_path.exists():
    import shutil
    shutil.copy(template_path, config_path)  # รันครั้งแรกยังไม่มีไฟล์ตั้งค่า สร้างจากต้นแบบก่อน
if config_path.exists():
    config = json.loads(config_path.read_text(encoding='utf-8'))
    config.setdefault('lang', {})
    config['lang']['override'] = True
    config['lang']['selected_lang'] = language
    config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

# ปลั๊กอิน: วางลิงก์ Hugging Face แล้วเลือกโหลดโมเดลหลายตัวพร้อมกัน (แท็บ Plugins > MultiModelDownload)
plugin_dir = repo_path / 'tabs' / 'plugins' / 'installed' / 'MultiModelDownload'
plugin_dir.mkdir(parents=True, exist_ok=True)
(plugin_dir / '__init__.py').write_text('', encoding='utf-8')
(plugin_dir / 'plugin.py').write_text('# ปลั๊กอิน Applio: วางลิงก์ Hugging Face แล้วเลือกโหลดโมเดลหลายตัวพร้อมกัน\nimport os\nimport re\nimport shutil\nimport requests\nimport gradio as gr\n\nnow_dir = os.getcwd()\nlogs_dir = os.path.join(now_dir, "logs")\nzips_dir = os.path.join(logs_dir, "zips")\ndrive_backup = "/content/drive/MyDrive/ApplioBackup"\n\n\ndef parse_link(link):\n    m = re.match(r"https?://huggingface\\.co/([^/\\s]+/[^/\\s]+)(?:/(?:tree|blob|resolve)/([^/\\s]+)/?(.*))?", link.strip())\n    if not m:\n        raise gr.Error("ลิงก์ไม่ถูกต้อง ต้องเป็นลิงก์ huggingface.co เช่น https://huggingface.co/ชื่อ/ชื่อโมเดล/tree/main")\n    return m.group(1), m.group(2) or "main", (m.group(3) or "").strip("/")\n\n\ndef list_files(link):\n    repo, rev, sub = parse_link(link)\n    url = f"https://huggingface.co/api/models/{repo}/tree/{rev}" + (f"/{sub}" if sub else "") + "?recursive=1"\n    r = requests.get(url, timeout=30)\n    if r.status_code != 200:\n        raise gr.Error(f"เปิดลิงก์ไม่ได้ (รหัส {r.status_code}) ตรวจว่าลิงก์ถูกและเป็นแบบสาธารณะ")\n    files = [f for f in r.json() if f.get("type") == "file" and f["path"].lower().endswith(".zip")]\n    if not files:\n        raise gr.Error("ไม่พบไฟล์ .zip ในลิงก์นี้")\n    choices = [(f"{f[\'path\']}  ({f.get(\'size\', 0) / 1024 / 1024:.0f} MB)", f["path"]) for f in files]\n    return (\n        gr.update(choices=choices, value=[], visible=True),\n        {"repo": repo, "rev": rev, "_all": [f["path"] for f in files]},\n        f"พบ {len(files)} โมเดล ติ๊กเลือกตัวที่ต้องการ แล้วกด \\"ดาวน์โหลดที่เลือก\\"",\n    )\n\n\ndef download_selected(selected, source, progress=gr.Progress()):\n    from rvc.lib.tools.model_download import handle_extraction_process\n\n    if not source:\n        raise gr.Error("กรุณากด \\"แสดงรายการโมเดล\\" ก่อน")\n    if not selected:\n        raise gr.Error("กรุณาติ๊กเลือกอย่างน้อย 1 โมเดล")\n    os.makedirs(zips_dir, exist_ok=True)\n    report = []\n    for i, path in enumerate(selected):\n        name = os.path.basename(path)\n        progress(i / len(selected), desc=f"กำลังโหลด {name} ({i + 1}/{len(selected)})")\n        try:\n            url = f"https://huggingface.co/{source[\'repo\']}/resolve/{source[\'rev\']}/{path}"\n            with requests.get(url, stream=True, timeout=60) as r:\n                r.raise_for_status()\n                with open(os.path.join(zips_dir, name), "wb") as f:\n                    for chunk in r.iter_content(1024 * 1024):\n                        f.write(chunk)\n            result = handle_extraction_process()\n            if result == "Error":\n                raise RuntimeError("แตกไฟล์ zip ไม่สำเร็จ")\n            keep_in_drive(os.path.splitext(name)[0])\n            report.append(f"✅ {name}")\n        except Exception as e:\n            report.append(f"❌ {name} : {e}")\n    progress(1, desc="เสร็จแล้ว")\n    ok = sum(r.startswith("✅") for r in report)\n    return f"โหลดสำเร็จ {ok} จาก {len(selected)} โมเดล\\n" + "\\n".join(report) + \\\n        "\\n\\nไปที่แท็บแปลงเสียง แล้วกดปุ่มรีเฟรชรายการโมเดล จะเห็นโมเดลใหม่"\n\n\ndef keep_in_drive(zip_stem):\n    # ถ้าเชื่อม Google Drive ไว้ ย้ายโมเดลไปเก็บใน Drive แล้วทำทางลัดไว้ที่เดิม (รอบหน้าไม่ต้องโหลดใหม่)\n    if not os.path.isdir(drive_backup):\n        return\n    from rvc.lib.utils import format_title\n    folder = os.path.join(logs_dir, format_title(zip_stem))\n    target = os.path.join(drive_backup, os.path.basename(folder))\n    if os.path.isdir(folder) and not os.path.islink(folder) and not os.path.exists(target):\n        shutil.move(folder, target)\n        os.symlink(target, folder)\n\n\ndef applio_plugin():\n    gr.Markdown("## โหลดโมเดลหลายตัวพร้อมกัน\\nวางลิงก์ Hugging Face (หน้าที่มีไฟล์ .zip หลายไฟล์) แล้วกด \\"แสดงรายการโมเดล\\"")\n    source = gr.State({})\n    with gr.Row():\n        link = gr.Textbox(label="ลิงก์ Hugging Face", placeholder="https://huggingface.co/ชื่อ/ชื่อโมเดล/tree/main", scale=4)\n        list_btn = gr.Button("แสดงรายการโมเดล", variant="primary", scale=1)\n    picks = gr.CheckboxGroup(label="เลือกโมเดลที่จะโหลด (เลือกได้หลายอัน)", choices=[], visible=False)\n    with gr.Row():\n        all_btn = gr.Button("เลือกทั้งหมด")\n        none_btn = gr.Button("ล้างที่เลือก")\n        dl_btn = gr.Button("ดาวน์โหลดที่เลือก", variant="primary")\n    status = gr.Textbox(label="สถานะ", lines=8, interactive=False)\n\n    list_btn.click(list_files, [link], [picks, source, status])\n    link.submit(list_files, [link], [picks, source, status])\n    all_btn.click(lambda s: gr.update(value=s.get("_all", [])), [source], [picks])\n    none_btn.click(lambda: gr.update(value=[]), None, [picks])\n    dl_btn.click(download_selected, [picks, source], [status])\n', encoding='utf-8')

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs --bind_all

if sharing_method == 'gradio':
    !python app.py --listen --share --client
elif sharing_method == 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !lt --port 6969 & python app.py --listen --client
elif sharing_method == 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f'Ngrok URL: {listener.url()}')
    !python app.py --listen --client
